# NB04 — Flash Attention from Scratch

**North star callback:** Flash attention is Unsloth's single biggest optimization.
It eliminates the O(N²) memory footprint of standard attention by never materializing
the full N×N attention matrix. Instead, it computes attention tile by tile in SRAM.

We derive the math, implement a forward kernel in Triton, then read Unsloth's version.

## 1. The problem with naive attention

Standard scaled dot-product attention:
```
A = softmax(QKᵀ / √d) · V
```

For sequence length N and head dim d:
- QKᵀ produces an N×N matrix → **O(N²) memory**
- At N=8192, d=128: 8192² × 2 bytes = **128 MB per head per layer**
- Llama 3 8B has 32 heads × 32 layers = 131 GB just for attention maps
- This is why naive attention OOMs at long sequences

**Solution:** never store the full N×N matrix. Compute it in tiles.

In [1]:
import torch
import torch.nn.functional as F

def naive_attention(Q, K, V, scale=None):
    """Standard attention — materializes full N×N matrix."""
    if scale is None:
        scale = Q.shape[-1] ** -0.5
    scores = torch.matmul(Q, K.transpose(-2, -1)) * scale  # (B, H, N, N)
    attn = F.softmax(scores, dim=-1)
    return torch.matmul(attn, V)

B, H, N, d = 2, 8, 1024, 64
Q = torch.randn(B, H, N, d, device="cuda", dtype=torch.bfloat16)
K = torch.randn(B, H, N, d, device="cuda", dtype=torch.bfloat16)
V = torch.randn(B, H, N, d, device="cuda", dtype=torch.bfloat16)

torch.cuda.reset_peak_memory_stats()
out_ref = naive_attention(Q, K, V)
naive_vram = torch.cuda.max_memory_allocated() / 1024**2
print(f"Naive attention VRAM: {naive_vram:.1f} MB  (N={N}, H={H})")

Naive attention VRAM: 80.1 MB  (N=1024, H=8)


## 2. Online softmax — the key insight

To tile the softmax, we need to compute it incrementally.
For a sequence x₁, x₂, ..., xₙ we maintain:

- `m` = running max so far
- `l` = running sum of exp(xᵢ - m)

When we see a new block with max `m_new`:
```
m_new = max(m, block_max)
l_new = l * exp(m - m_new) + sum(exp(block - m_new))
```

After all blocks, the final softmax weight for xᵢ is `exp(xᵢ - m) / l`.
This is numerically identical to standard softmax, but requires only one pass.

In [2]:
import torch, math

def online_softmax_tiled(x: torch.Tensor, BLOCK: int = 64) -> torch.Tensor:
    """Online softmax — single pass over tiles, numerically stable."""
    m = torch.tensor(float("-inf"), dtype=x.dtype)
    l = torch.tensor(0.0, dtype=x.dtype)
    N = x.shape[0]
    for i in range(0, N, BLOCK):
        block = x[i:i+BLOCK]
        m_block = block.max()
        m_new = torch.maximum(m, m_block)
        l = l * (m - m_new).exp() + (block - m_new).exp().sum()
        m = m_new
    return (x - m).exp() / l

x = torch.randn(512)
ref = torch.softmax(x, dim=0)
got = online_softmax_tiled(x)
assert torch.allclose(got, ref, atol=1e-5)
print("✓ online softmax (tiled) matches reference")

✓ online softmax (tiled) matches reference


In [3]:
import triton
import triton.language as tl


@triton.jit
def flash_attn_forward_kernel(
    Q_ptr, K_ptr, V_ptr, Out_ptr,
    L_ptr,          # log-sum-exp for backward pass
    stride_qb, stride_qh, stride_qn, stride_qd,
    stride_kb, stride_kh, stride_kn, stride_kd,
    stride_vb, stride_vh, stride_vn, stride_vd,
    stride_ob, stride_oh, stride_on, stride_od,
    N, d,
    scale,
    BLOCK_N: tl.constexpr,
    BLOCK_D: tl.constexpr,
):
    """Flash attention forward — one program handles one (batch, head, query-tile)."""
    # Program coordinates
    start_n = tl.program_id(0) * BLOCK_N  # query tile start
    off_bh = tl.program_id(1)             # (batch, head) index

    # Pointers to Q tile
    offs_n = start_n + tl.arange(0, BLOCK_N)
    offs_d = tl.arange(0, BLOCK_D)
    mask_n = offs_n < N

    q_ptrs = Q_ptr + off_bh * stride_qb + offs_n[:, None] * stride_qn + offs_d[None, :] * stride_qd
    # Load Q and upcast to float32 — all accumulation done in float32 for stability
    Q = tl.load(q_ptrs, mask=mask_n[:, None], other=0.0).to(tl.float32)  # (BLOCK_N, BLOCK_D)

    # Accumulators
    m_i = tl.full((BLOCK_N,), float("-inf"), dtype=tl.float32)  # running max
    l_i = tl.zeros((BLOCK_N,), dtype=tl.float32)                 # running sum
    O = tl.zeros((BLOCK_N, BLOCK_D), dtype=tl.float32)           # output accumulator

    # Iterate over key-value tiles
    for start_k in range(0, N, BLOCK_N):
        offs_k = start_k + tl.arange(0, BLOCK_N)
        mask_k = offs_k < N

        k_ptrs = K_ptr + off_bh * stride_kb + offs_k[None, :] * stride_kn + offs_d[:, None] * stride_kd
        K = tl.load(k_ptrs, mask=mask_k[None, :], other=0.0).to(tl.float32)  # (BLOCK_D, BLOCK_N)

        # QKᵀ for this tile — both float32, result is float32
        S = tl.dot(Q, K) * scale  # (BLOCK_N, BLOCK_N)

        # Online softmax update
        m_block = tl.max(S, axis=1)
        m_new = tl.maximum(m_i, m_block)
        alpha = tl.exp(m_i - m_new)
        beta = tl.exp(S - m_new[:, None])
        l_i = l_i * alpha + tl.sum(beta, axis=1)
        O = O * alpha[:, None]

        v_ptrs = V_ptr + off_bh * stride_vb + offs_k[:, None] * stride_vn + offs_d[None, :] * stride_vd
        V = tl.load(v_ptrs, mask=mask_k[:, None], other=0.0).to(tl.float32)  # (BLOCK_N, BLOCK_D)
        O = O + tl.dot(beta, V)
        m_i = m_new

    # Normalize
    O = O / l_i[:, None]

    # Store log-sum-exp for backward
    l_ptrs = L_ptr + off_bh * N + offs_n
    tl.store(l_ptrs, m_i + tl.log(l_i), mask=mask_n)

    # Store output — cast back to bfloat16
    out_ptrs = Out_ptr + off_bh * stride_ob + offs_n[:, None] * stride_on + offs_d[None, :] * stride_od
    tl.store(out_ptrs, O.to(tl.bfloat16), mask=mask_n[:, None])


def flash_attention_forward(Q, K, V):
    B, H, N, d = Q.shape
    BLOCK_N = 64
    BLOCK_D = triton.next_power_of_2(d)
    scale = d ** -0.5

    # Flatten (B, H) into one dim for kernel dispatch
    Q_f = Q.reshape(B * H, N, d).contiguous()
    K_f = K.reshape(B * H, N, d).contiguous()
    V_f = V.reshape(B * H, N, d).contiguous()
    Out = torch.empty_like(Q_f)
    L = torch.empty(B * H, N, device=Q.device, dtype=torch.float32)

    grid = (triton.cdiv(N, BLOCK_N), B * H)
    flash_attn_forward_kernel[grid](
        Q_f, K_f, V_f, Out, L,
        Q_f.stride(0), 0, Q_f.stride(1), Q_f.stride(2),
        K_f.stride(0), 0, K_f.stride(1), K_f.stride(2),
        V_f.stride(0), 0, V_f.stride(1), V_f.stride(2),
        Out.stride(0), 0, Out.stride(1), Out.stride(2),
        N, d, scale,
        BLOCK_N=BLOCK_N, BLOCK_D=BLOCK_D,
    )
    return Out.reshape(B, H, N, d), L

In [4]:
out_flash, L = flash_attention_forward(Q, K, V)

# Reference (PyTorch SDPA)
out_ref = F.scaled_dot_product_attention(Q, K, V)
max_err = (out_flash.float() - out_ref.float()).abs().max().item()
print(f"Max error vs SDPA: {max_err:.6f}  (should be < 0.01 for bf16)")

import sys; sys.path.insert(0, "..")
from utils.benchmark import compare

results = compare(
    fns={
        "naive": lambda: naive_attention(Q, K, V),
        "pytorch_sdpa": lambda: F.scaled_dot_product_attention(Q, K, V),
        "flash_triton": lambda: flash_attention_forward(Q, K, V)[0],
    },
    notebook="nb04",
    experiment="attention_N1024",
    n_warmup=10, n_repeat=50,
)
for label, r in results.items():
    print(f"{label}: {r.latency_ms:.3f} ms  |  {r.peak_vram_mb:.1f} MB VRAM")

Max error vs SDPA: 0.001953  (should be < 0.01 for bf16)
naive: 0.104 ms  |  84.2 MB VRAM
pytorch_sdpa: 0.035 ms  |  20.2 MB VRAM
flash_triton: 0.082 ms  |  20.2 MB VRAM


## 5. Read Unsloth's flash attention

Unsloth's attention kernels live in `unsloth/kernels/fast_lora.py` and the
model-specific files (e.g. `unsloth/models/llama.py`). Open them:

```python
import inspect, unsloth
import unsloth.kernels
print(unsloth.kernels.__file__)
```

Key differences from our implementation:
1. **Causal masking** — the upper triangle is masked out
2. **GQA support** — grouped-query attention (different K/V head counts)
3. **ALiBi/RoPE integration** — position biases applied in-kernel
4. **fp8 / bf16 mixed precision** — careful dtype handling

In [5]:
import sys
sys.path.insert(0, "..")
# unsloth is an editable install — add its source tree so kernels subpackage is importable
sys.path.insert(0, "../unsloth")
import unsloth.kernels
print("Unsloth kernels location:", unsloth.kernels.__file__)
# Navigate to the attention forward kernel and read it

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


Unsloth kernels location: /home/rob/Data3/PythonEnvironments/UnslothLab/notebooks/../unsloth/unsloth/kernels/__init__.py


## 6. Exercises

1. **Increase N**: run the benchmark with N=2048 and N=4096.
   At what sequence length does naive attention OOM?

2. **Add causal masking**: modify `flash_attn_forward_kernel` to set S to -inf
   for positions where `offs_k > offs_n`. Verify output matches
   `F.scaled_dot_product_attention(Q, K, V, is_causal=True)`.

3. **Measure SRAM usage**: change BLOCK_N from 64 to 128.
   What happens to speed? Why does SRAM capacity limit the block size?